In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

RESULTS_DIR = "../results"
PLOTS_DIR   = "../plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
#Load memory data
memory_df = pd.read_csv(f"{RESULTS_DIR}/memory_results.csv")
print(memory_df)

In [ ]:
# Load accuracy data and compute average score per variant
def load_eval(path):
    with open(path) as f:
        data = json.load(f)
    r = data["results"]
    return {
        "mmlu":       r["mmlu"]["acc"],
        "hellaswag":  r["hellaswag"]["acc_norm"],
        "gsm8k":      r["gsm8k"]["acc"],
        "humaneval":  r["humaneval"]["pass@1"],
        "truthfulqa": r["truthfulqa_mc1"]["acc"],
    }

scores = {
    "fp16": load_eval(f"{RESULTS_DIR}/accuracy/fp16_eval.json"),
    "int8": load_eval(f"{RESULTS_DIR}/accuracy/int8_eval.json"),
    "gptq": load_eval(f"{RESULTS_DIR}/accuracy/gptq_eval.json"),
    "awq":  load_eval(f"{RESULTS_DIR}/accuracy/awq_eval.json"),
}

accuracy_df = pd.DataFrame(scores).T
accuracy_df["avg_accuracy"] = accuracy_df.mean(axis=1)
print(accuracy_df[["avg_accuracy"]])

In [ ]:
# Build combined dataframe for Pareto chart
pareto_df = pd.DataFrame({
    "model":        memory_df.index,
    "peak_mem_gb":  memory_df["peak_2k_gb"].values,
    "avg_accuracy": accuracy_df["avg_accuracy"].values,
})
print(pareto_df)

In [ ]:
# Pareto scatter plot
# X = memory, Y = accuracy
# Ideal point = bottom left (low memory, high accuracy)
colors = {"fp16": "#2196F3", "int8": "#4CAF50", "gptq": "#FF9800", "awq": "#E91E63"}

fig, ax = plt.subplots(figsize=(10, 7))

for _, row in pareto_df.iterrows():
    ax.scatter(
        row["peak_mem_gb"],
        row["avg_accuracy"],
        color=colors.get(row["model"], "gray"),
        s=200,
        zorder=5
    )
    ax.annotate(
        row["model"].upper(),
        (row["peak_mem_gb"], row["avg_accuracy"]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=12,
        fontweight="bold"
    )

ax.set_xlabel("Peak GPU Memory at 2K Context (GB)", fontsize=13)
ax.set_ylabel("Average Accuracy across 5 Benchmarks", fontsize=13)
ax.set_title("Pareto Frontier: Memory vs Accuracy\n(closer to top-left = better tradeoff)",
             fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/pareto_memory_vs_accuracy.png", dpi=150)
plt.show()
print("Saved pareto_memory_vs_accuracy.png — this is your headline chart")

In [ ]:
# Decision matrix
# The "when would I use what" summary table
decision_matrix = pd.DataFrame([
    {
        "Constraint":      "Memory is critical (consumer GPU)",
        "Recommendation":  "GPTQ or AWQ 4-bit",
        "Why":             "4x smaller than FP16"
    },
    {
        "Constraint":      "Accuracy is paramount",
        "Recommendation":  "FP16 or INT8",
        "Why":             "Minimal precision loss"
    },
    {
        "Constraint":      "Throughput at scale",
        "Recommendation":  "AWQ",
        "Why":             "Best optimized inference kernels"
    },
    {
        "Constraint":      "Math or code heavy workload",
        "Recommendation":  "INT8 or FP16",
        "Why":             "4-bit degrades reasoning tasks"
    },
    {
        "Constraint":      "Quick experiment, no calibration",
        "Recommendation":  "bitsandbytes INT8",
        "Why":             "Zero config, just load_in_8bit=True"
    },
])

print("\nDecision Matrix — When to use what:")
print(decision_matrix.to_string(index=False))

In [ ]:
# Summary stats
# Final numbers to put in your writeup
print("\n=== Final Summary ===")
print(f"\nMemory savings vs FP16:")
fp16_mem = memory_df.loc["fp16", "peak_2k_gb"]
for model in ["int8", "gptq", "awq"]:
    mem = memory_df.loc[model, "peak_2k_gb"]
    savings = fp16_mem / mem
    print(f"  {model}: {mem:.2f} GB ({savings:.1f}x smaller)")

print(f"\nAverage accuracy vs FP16:")
fp16_acc = accuracy_df.loc["fp16", "avg_accuracy"]
for model in ["int8", "gptq", "awq"]:
    acc = accuracy_df.loc[model, "avg_accuracy"]
    drop = (fp16_acc - acc) * 100
    print(f"  {model}: {acc:.3f} ({drop:.1f} points below FP16)")